# Experiment with Entra brokered authentication

In [1]:
import base64
import json
import sys
from datetime import datetime

import requests

from azure.identity import InteractiveBrowserCredential
from azure.identity.broker import InteractiveBrowserBrokerCredential

In [2]:
scope = 'https://graph.microsoft.com/.default'

In [3]:
def decode_access_token(access_token):
    segments = access_token.split('.')
    claims_base64 = segments[1] + ('=' * (3 - ((len(segments[1]) + 3) % 4)))
    claims_string = base64.urlsafe_b64decode(claims_base64).decode('utf-8')
    claims = json.loads(claims_string)
    claims['exp'] = datetime.fromtimestamp(claims['exp'])
    claims['nbf'] = datetime.fromtimestamp(claims['nbf'])
    claims['iat'] = datetime.fromtimestamp(claims['iat'])
    return claims

In [4]:
if sys.platform == "win32":
    import win32gui
    window_handle=win32gui.GetForegroundWindow()
else:
    import msal
    window_handle=msal.PublicClientApplication.CONSOLE_WINDOW_HANDLE

## Test interactive auth without an auth broker

In [6]:
decode_access_token(InteractiveBrowserCredential().get_token(scope).token)

InteractiveBrowserCredential.get_token failed: Authentication failed: access_denied


ClientAuthenticationError: Authentication failed: access_denied

Auth pop-up error:
> **Sorry, a security policy is preventing acces**
>
> An organization security policy requiring token protection is preventing this application from accessing the resource. You may be able to use a different application.

## Test interactive auth with an auth broker

In [7]:
def test_app(client_id):
    credential = InteractiveBrowserBrokerCredential(parent_window_handle=window_handle, client_id=client_id)
    access_token = credential.get_token(scope).token
    response = requests.get(
        f"https://graph.microsoft.com/v1.0/servicePrincipals?$filter=appId eq '{client_id}'",
        headers={"Authorization": "Bearer " + access_token}
    )
    response.raise_for_status()
    print('> App name:')
    print(response.json()['value'][0]['displayName'])
    print()
    print('> Access token scopes:')
    print(decode_access_token(access_token)['scp'].replace(' ', '\n'))

In [32]:
test_app('04b07795-8ddb-461a-bbee-02f9e1bf7b46')

> App name:
Microsoft Azure CLI

> Access token scopes:
Application.ReadWrite.All
AppRoleAssignment.ReadWrite.All
AuditLog.Read.All
DelegatedPermissionGrant.ReadWrite.All
Directory.AccessAsUser.All
email
Group.ReadWrite.All
openid
profile
User.Read.All
User.ReadWrite.All


In [33]:
test_app('d3590ed6-52b3-4102-aeff-aad2292ab01c')

> App name:
Microsoft Office

> Access token scopes:
AuditLog.Create
Calendar.ReadWrite
Calendars.Read.Shared
Calendars.ReadWrite
Contacts.ReadWrite
DataLossPreventionPolicy.Evaluate
Directory.AccessAsUser.All
Directory.Read.All
email
Files.Read
Files.Read.All
Files.ReadWrite.All
FileStorageContainer.Selected
Group.Read.All
Group.ReadWrite.All
InformationProtectionPolicy.Read
Mail.ReadWrite
Mail.Send
Notes.Create
openid
Organization.Read.All
People.Read
People.Read.All
Printer.Read.All
PrinterShare.ReadBasic.All
PrintJob.Create
PrintJob.ReadWriteBasic
profile
Reports.Read.All
SensitiveInfoType.Detect
SensitiveInfoType.Read.All
SensitivityLabel.Evaluate
Tasks.ReadWrite
TeamMember.ReadWrite.All
TeamsTab.ReadWriteForChat
User.Read.All
User.ReadBasic.All
User.ReadWrite
Users.Read


In [8]:
test_app('ba081686-5d24-4bc6-a0d6-d034ecffed87')

> App name:
Work IQ CLI

> Access token scopes:
ChannelMessage.Read.All
Chat.Read
ExternalItem.Read.All
Mail.Read
OnlineMeetingTranscript.Read.All
People.Read.All
Sites.Read.All
profile
openid
email


In [9]:
test_app('1950a258-227b-4e31-a9cf-717495945fc2')

> App name:
Microsoft Azure PowerShell

> Access token scopes:
Application.ReadWrite.All
AppRoleAssignment.ReadWrite.All
AuditLog.Read.All
DelegatedPermissionGrant.ReadWrite.All
Directory.AccessAsUser.All
email
Group.ReadWrite.All
openid
profile
User.Read.All


In [11]:
test_app('1fec8e78-bce4-4aaf-ab1b-5451cc387264')

> App name:
Microsoft Teams

> Access token scopes:
AppCatalog.Read.All
Calendars.Read
Calendars.Read.Shared
Calendars.ReadWrite
Calendars.ReadWrite.Shared
Channel.ReadBasic.All
ChatMessage.Send
Contacts.ReadWrite.Shared
email
Files.ReadWrite.All
FileStorageContainer.Selected
InformationProtectionPolicy.Read
Mail.ReadWrite
Mail.Send
MailboxSettings.ReadWrite
Notes.ReadWrite.All
openid
Organization.Read.All
People.Read
Place.Read.All
profile
Sites.ReadWrite.All
Tasks.ReadWrite
Team.ReadBasic.All
TeamsAppInstallation.ReadForTeam
TeamsTab.Create
User.ReadBasic.All
